# Novel Findings — AI vs. Human Bug-Fix Pull Requests

**Data:** closed bug-fix PRs, Dec 2024 – Feb 2026 (15 months), agents Copilot / Cursor / Claude Code / Devin vs. human authors.
Replication of basic metrics (acceptance, speed, size) is in themes 1–3. **This notebook holds the new findings.**

Five questions:
1. **How does each quality metric change over time?**
2. **Do repositories change which agent they use over time?**
3. **Do developers add/update agent-instruction files over time?**
4. **Do agents fix different *kinds* of bugs than humans?**
5. **Do agent fixes hold up?** (revert rate, with a fairness control)

Every finding below states a plain answer and an honest caveat.

In [1]:
import sys; sys.path.insert(0, '.')
from pathlib import Path
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, classify_file_role, classify_bug_type, BUG_CATEGORIES,
    set_plot_style, save_fig, AGENTS, AGENT_COLORS, MIN_N_PROP, MIN_N_MEDIAN,
)
import re, numpy as np, pandas as pd, matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()
FIG = Path('results/findings_figures')      # all figures from this notebook land here
CATS = list(BUG_CATEGORIES) + ['other']

# Load once
df = load_fix_prs()
df['bug_type'] = df['title'].map(classify_bug_type)
agents_df = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df  = df[~df['is_agent']].copy()
commits   = load_commits()
details   = load_commit_details()
rev_stats = build_revision_stats(df, commits, details)
agent_rev = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
details['role'] = details['filename'].map(classify_file_role)
prod = details[details.role=='prod'].groupby('pr_id')['additions'].sum().rename('prod_added')
df_size = df.merge(prod, left_on='id', right_index=True, how='left')
months = sorted(df['month'].unique()); idx = [str(m) for m in months]
print('Loaded:', f'{len(df):,} fix PRs across {len(months)} months')

Loading fix PRs from HuggingFace ...


  AIDev repo coverage: 8,959 distinct repos
  Survivorship cutoff at 2026-01-29: dropped 33,123 recent PRs
  Fix PRs loaded: 371,577  |  Agent: 108,080  |  Human: 263,497


Loading commits from HuggingFace ...


  Commits loaded: 1,156,238
Loading commit details from HuggingFace ...


  Commit details loaded: 7,451,150


Loaded: 371,577 fix PRs across 14 months


## Finding 1 — Quality trajectories *diverge* between agents over time

We track four metrics month by month for each agent (acceptance, time-to-merge, production lines, revision rate). Months with too few PRs are dropped (n>=30 rates, n>=20 medians).

In [2]:
def monthly(agent, metric):
    out=[]
    for m in months:
        if metric=='acceptance':
            s=agents_df[(agents_df.month==m)&(agents_df.agent==agent)]
            out.append(s.is_merged.mean()*100 if len(s)>=MIN_N_PROP else np.nan)
        elif metric=='ttm':
            s=agents_df[(agents_df.month==m)&(agents_df.agent==agent)&agents_df.is_merged]
            out.append(s.hours_to_merge.median() if len(s)>=MIN_N_MEDIAN else np.nan)
        elif metric=='prodsize':
            s=df_size[(df_size.month==m)&(df_size.agent==agent)&df_size.is_agent]
            out.append(s.prod_added.median() if len(s)>=MIN_N_MEDIAN else np.nan)
        elif metric=='revision':
            s=agent_rev[(agent_rev.month==m)&(agent_rev.agent==agent)]
            out.append((s.num_commits>1).mean()*100 if len(s)>=MIN_N_PROP else np.nan)
    return out

PANELS=[('acceptance','Acceptance %'),('ttm','Median time-to-merge (h)'),
        ('prodsize','Median production lines'),('revision','Revision rate %')]
fig, axes = plt.subplots(2,2,figsize=(14,9)); x=list(range(len(idx)))
for ax,(mk,lbl) in zip(axes.ravel(),PANELS):
    for a in AGENTS:
        ax.plot(idx, monthly(a,mk),'o-',color=AGENT_COLORS[a],label=a,linewidth=1.6,markersize=4)
    ax.axvline('2025-07',color='red',linestyle=':',linewidth=1.2)
    ax.set_title(lbl); ax.set_xticks(x[::2]); ax.set_xticklabels(idx[::2],rotation=45,ha='right',fontsize=8)
axes.ravel()[0].legend(fontsize=8,ncol=2)
fig.suptitle('Finding 1: Bug-fix quality metrics over time, per agent',fontsize=13)
fig.tight_layout(); save_fig(fig,'f1_metrics_over_time',FIG)

# quick numeric summary (first vs last reported month)
print('Acceptance %, first -> last reported month:')
for a in AGENTS:
    s=[v for v in monthly(a,'acceptance') if not np.isnan(v)]
    print(f'  {a:<12} {s[0]:.0f}% -> {s[-1]:.0f}%')

  -> Saved: results\findings_figures\f1_metrics_over_time.png
Acceptance %, first -> last reported month:
  Copilot      91% -> 76%
  Cursor       95% -> 89%
  Claude_Code  83% -> 90%
  Devin        58% -> 77%


![f1_metrics_over_time](results/findings_figures/f1_metrics_over_time.png)

**Finding:** the agents do **not** move together. **Claude Code's acceptance rises (~83%→90%)**, while **Copilot (~91%→76%) and Cursor decline**; production patch sizes drift up for most agents.
**Why it's trustworthy:** agents move in *opposite* directions, so this is not a data artifact (a collection artifact would push all agents the same way).
**Caveat:** read the last 1–2 months gently (recent slow merges may not be finished); trends are descriptive, not causal.

## Finding 2 — Repositories mostly stay loyal to one agent

In [3]:
per_agent = (agents_df.groupby(['month','agent']).size().unstack(fill_value=0)
             .reindex(columns=AGENTS, fill_value=0))
share = per_agent.div(per_agent.sum(axis=1), axis=0)*100

g = agents_df.groupby(['repo','month','agent']).size().reset_index(name='n').sort_values(['repo','month','n'])
dom = g.drop_duplicates(['repo','month'], keep='last')[['repo','month','agent']]
multi = dom.groupby('repo').filter(lambda x: x['month'].nunique()>=2)
switched = multi.groupby('repo')['agent'].nunique().gt(1).sum(); n_multi = multi['repo'].nunique()
print(f'Repos active in >=2 months: {n_multi:,}')
print(f'  kept same main agent : {n_multi-switched:,} ({100*(n_multi-switched)/n_multi:.1f}%)')
print(f'  switched main agent  : {switched:,} ({100*switched/n_multi:.1f}%)')

fig, ax = plt.subplots(figsize=(13,5))
for a in AGENTS: ax.plot(idx, share[a].values,'o-',color=AGENT_COLORS[a],label=a,linewidth=1.8)
ax.axvline('2025-07',color='red',linestyle=':',linewidth=1.2,label='AIDev cutoff')
ax.set_ylabel('Share of agent bug-fix PRs (%)'); ax.set_title('Finding 2: Agent share over time')
plt.xticks(rotation=45,ha='right'); ax.legend(); fig.tight_layout()
save_fig(fig,'f2_agent_share_over_time',FIG)

Repos active in >=2 months: 3,222
  kept same main agent : 2,743 (85.1%)
  switched main agent  : 479 (14.9%)
  -> Saved: results\findings_figures\f2_agent_share_over_time.png


WindowsPath('results/findings_figures/f2_agent_share_over_time.png')

![f2_agent_share_over_time](results/findings_figures/f2_agent_share_over_time.png)

**Finding:** **~85% of repositories keep the same main agent**; only ~15% switch. Aggregate share shifts over time, but that is new repos arriving, not existing repos changing tools.
**Caveat:** "switching" is measured per **repository** (agent PRs come from the agent's bot account, so we can't track individual developers).

## Finding 3 — Use of agent-instruction files is rising

In [4]:
instr_pat = r'CLAUDE\.md|AGENTS?\.md|GEMINI\.md|\.cursorrules|\.cursor/rules/|copilot-instructions|\.windsurfrules|\.clinerules'
details['is_instr'] = details['filename'].str.contains(instr_pat, case=False, regex=True, na=False)
df['touches_instr'] = df['id'].isin(set(details.loc[details.is_instr,'pr_id']))
monthly_instr = df[df.touches_instr].groupby('month').size().reindex(months, fill_value=0)
cum_repos = (df[df.touches_instr].groupby('repo')['month'].min()
             .value_counts().reindex(months, fill_value=0).cumsum())
print(f'Fix PRs touching an instruction file: {df.touches_instr.sum():,} ({df.touches_instr.mean()*100:.2f}%), '
      f'{df.loc[df.touches_instr,"repo"].nunique():,} repos')
instr_repos=set(df.loc[df.touches_instr,'repo'])
print(f'Agent acceptance: repos WITH instr files {agents_df[agents_df.repo.isin(instr_repos)].is_merged.mean()*100:.1f}% '
      f'vs WITHOUT {agents_df[~agents_df.repo.isin(instr_repos)].is_merged.mean()*100:.1f}%')

fig, ax = plt.subplots(figsize=(13,5)); x=list(range(len(idx)))
ax.bar(x, monthly_instr.values, color='#2ca02c', alpha=0.6)
ax2=ax.twinx(); ax2.plot(x, cum_repos.values,'o-',color='#1f77b4')
ax.set_xticks(x[::2]); ax.set_xticklabels(idx[::2],rotation=45,ha='right',fontsize=8)
ax.set_ylabel('Fix PRs touching instr files (monthly)',color='#2ca02c')
ax2.set_ylabel('Cumulative repos',color='#1f77b4')
ax.set_title('Finding 3: Adoption of agent-instruction files over time')
fig.tight_layout(); save_fig(fig,'f3_instruction_files',FIG)

Fix PRs touching an instruction file: 2,222 (0.60%), 837 repos
Agent acceptance: repos WITH instr files 85.7% vs WITHOUT 82.9%
  -> Saved: results\findings_figures\f3_instruction_files.png


WindowsPath('results/findings_figures/f3_instruction_files.png')

![f3_instruction_files](results/findings_figures/f3_instruction_files.png)

**Finding:** developers increasingly add/update agent-instruction files (CLAUDE.md, .cursorrules, etc.); cumulative adopting repos grow steadily through 2025. Repos that use them have somewhat higher agent acceptance.
**Caveat:** lower bound — we only see instruction-file edits that happen inside a bug-fix PR. The acceptance gap is **associational** (such repos likely differ in maturity).

## Finding 4 — Agents and humans fix *different kinds* of bugs

In [5]:
a_share=agents_df.bug_type.value_counts(normalize=True).reindex(CATS).fillna(0)*100
h_share=human_df.bug_type.value_counts(normalize=True).reindex(CATS).fillna(0)*100
order=a_share.sort_values(ascending=False).index.tolist()
acc=[(c, agents_df[agents_df.bug_type==c].is_merged.mean()*100) for c in CATS
     if (agents_df.bug_type==c).sum()>=200]
acc=sorted(acc,key=lambda t:t[1])

fig,(ax1,ax2)=plt.subplots(1,2,figsize=(15,5))
x=np.arange(len(order)); w=0.4
ax1.bar(x-w/2,a_share[order].values,w,label='Agent',color='#1f77b4')
ax1.bar(x+w/2,h_share[order].values,w,label='Human',color='#7f7f7f')
ax1.set_xticks(x); ax1.set_xticklabels(order,rotation=35,ha='right'); ax1.legend()
ax1.set_ylabel("% of group's fixes"); ax1.set_title('Bug-type mix: agent vs human')
cs=[c for c,_ in acc]; vs=[v for _,v in acc]
ax2.barh(cs,vs,color=['#d62728' if c=='security' else '#1f77b4' for c in cs])
for i,v in enumerate(vs): ax2.text(v+0.3,i,f'{v:.0f}%',va='center',fontsize=8)
ax2.set_xlim(0,100); ax2.set_xlabel('Agent acceptance %'); ax2.set_title('Agent acceptance by bug type')
fig.tight_layout(); save_fig(fig,'f4_bug_types',FIG)
print(f'build/ci share  — agent {a_share["build/ci"]:.1f}% vs human {h_share["build/ci"]:.1f}%')
print(f'security acceptance (agents): {agents_df[agents_df.bug_type=="security"].is_merged.mean()*100:.1f}%')

  -> Saved: results\findings_figures\f4_bug_types.png
build/ci share  — agent 19.4% vs human 6.0%
security acceptance (agents): 71.3%


![f4_bug_types](results/findings_figures/f4_bug_types.png)

**Finding:** agents fix **~3× more build / CI / dependency bugs** than humans (the repetitive, well-scoped kind). And agent **security** fixes are **accepted least** (~71%), i.e. maintainers scrutinize them more.
**Caveat:** bug type is inferred from PR-title keywords (coarse; ~half are 'other'), so treat small categories loosely — but the build/CI gap is large enough to be robust.

## Finding 5 (preliminary) — Do agent fixes hold up? (reverts)

In [6]:
commits['message']=commits['message'].fillna('')
rev=commits[commits.message.str.contains('reverts commit',case=False,na=False)]
reverted=set(s[:12] for s in rev.message.str.extract(r'reverts commit ([0-9a-f]{7,40})',flags=re.I)[0].dropna())
commits['sha12']=commits['sha'].str[:12]
df['was_reverted']=df['id'].isin(set(commits.loc[commits.sha12.isin(reverted),'pr_id']))

# Fairness control: only PRs old enough to have had time to be reverted (created on/before 2025-07-31)
old = df[df['created_at'] <= pd.Timestamp('2025-07-31', tz='UTC')]
m_ag = old[old.is_merged & old.is_agent & old.agent.isin(AGENTS)]
m_hu = old[old.is_merged & ~old.is_agent]
print('Revert rate among merged fixes created on/before 2025-07-31 (>=6 months to be reverted):')
print(f'  Agents: {m_ag.was_reverted.mean()*100:.2f}%   Humans: {m_hu.was_reverted.mean()*100:.2f}%')

# within bug type (agent vs human)
rows=[]
for c in CATS:
    a=m_ag[m_ag.bug_type==c]; h=m_hu[m_hu.bug_type==c]
    if len(a)>=150 and len(h)>=150: rows.append((c,a.was_reverted.mean()*100,h.was_reverted.mean()*100))
ct=pd.DataFrame(rows,columns=['bug_type','agent%','human%'])
fig,ax=plt.subplots(figsize=(12,5)); x=np.arange(len(ct)); w=0.4
ax.bar(x-w/2,ct['agent%'],w,label='Agent',color='#1f77b4')
ax.bar(x+w/2,ct['human%'],w,label='Human',color='#7f7f7f')
ax.set_xticks(x); ax.set_xticklabels(ct.bug_type,rotation=35,ha='right'); ax.legend()
ax.set_ylabel('% of merged fixes later reverted')
ax.set_title('Finding 5: Revert rate within bug type (older PRs only)')
fig.tight_layout(); save_fig(fig,'f5_reverts',FIG)
print(ct.round(2).to_string(index=False))

Revert rate among merged fixes created on/before 2025-07-31 (>=6 months to be reverted):
  Agents: 0.22%   Humans: 0.49%


  -> Saved: results\findings_figures\f5_reverts.png
        bug_type  agent%  human%
     crash/error    0.18    0.46
  null/undefined    0.68    0.52
     typo/format    0.12    0.22
        security    0.10    0.29
     memory/leak    0.28    0.47
race/concurrency    0.16    0.71
      regression    0.00    0.52
        build/ci    0.16    0.63
      ui/display    0.18    0.76
           other    0.27    0.49


![f5_reverts](results/findings_figures/f5_reverts.png)

**Finding (preliminary):** even after restricting to older PRs (equal time to be reverted), **agent fixes are reverted less than human fixes within most bug types** — a tentative sign their merged fixes are more durable.
**Caveats (why preliminary):** (1) reverts are rare (~0.2–0.6%) and we only see reverts that occur inside a fix PR; (2) **repo mix is not controlled** — agents cluster in repos that may revert less. We do **not** yet claim "agents fix better"; a repo-matched test is the next step.

## Summary for the paper

**Replication (themes 1–3):** agents are accepted at similar/higher rates than humans, merge much faster, write somewhat larger patches.

**New contributions (this notebook):**
1. **Quality trajectories diverge** — Claude Code improves over time while Copilot/Cursor decline. (Static benchmarks can't see this.)
2. **Repos are agent-loyal** — ~85% never switch their main agent.
3. **Agent-instruction files are being adopted** and correlate with higher acceptance.
4. **Agents fix different bugs** — ~3× more build/CI; their security fixes are trusted least.
5. **(Preliminary)** agent fixes may be more durable (reverted less), pending a repo-matched control.

Headline story: *the value of 15 months of data — agent bug-fixing is not static; behavior, tooling, and quality shift over time, and agents specialize in a different slice of bugs than humans.*